## GAWD dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pavt/GHAW-H/blob/main/notebooks/load_GAWD.ipynb)

This notebook loads **A Dataset of GitHub Agentic Workflow Histories: Early Adopters** in the same lightweight style as `load_AIDev.ipynb`.

By default this notebook reads the public Parquet files from Hugging Face, matching how an external user would load the dataset. For local development, set `USE_LOCAL_MIRROR = True` to use a release folder with the same `data/*.parquet` layout.


In [1]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "pyarrow": "pyarrow",
}

if "google.colab" in sys.modules:
    missing = [
        package
        for package, import_name in REQUIRED_PACKAGES.items()
        if importlib.util.find_spec(import_name) is None
    ]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import pandas as pd

DATASET_ID = "pavtch/GHAW-H"
DATASET_REVISION = "main"
LOCAL_RELEASE_CANDIDATES = [
    Path(".."),
    Path("."),
    Path("../releases/gh-aw-early-adopters-source-history-2026-06-26"),
    Path("releases/gh-aw-early-adopters-source-history-2026-06-26"),
    Path("../releases/gawd"),
    Path("releases/gawd"),
]
LOCAL_RELEASE_MIRROR = next(
    (
        path.resolve()
        for path in LOCAL_RELEASE_CANDIDATES
        if (path / "data").is_dir() and any((path / "data").glob("*.parquet"))
    ),
    None,
)
USE_LOCAL_MIRROR = "google.colab" not in sys.modules and LOCAL_RELEASE_MIRROR is not None


def remote_parquet_url(table_name: str) -> str:
    return (
        f"https://huggingface.co/datasets/{DATASET_ID}/resolve/"
        f"{DATASET_REVISION}/data/{table_name}.parquet"
    )


def local_parquet_path(table_name: str) -> Path:
    if LOCAL_RELEASE_MIRROR is None:
        raise FileNotFoundError("No local release mirror with a data/ directory was found.")
    return LOCAL_RELEASE_MIRROR / "data" / f"{table_name}.parquet"


def parquet_path(table_name: str) -> str:
    if USE_LOCAL_MIRROR:
        return str(local_parquet_path(table_name))
    return remote_parquet_url(table_name)


def read_parquet_table(table_name: str) -> pd.DataFrame:
    if USE_LOCAL_MIRROR:
        return pd.read_parquet(local_parquet_path(table_name))

    try:
        with urlopen(remote_parquet_url(table_name)) as response:
            return pd.read_parquet(BytesIO(response.read()))
    except Exception:
        if LOCAL_RELEASE_MIRROR is not None:
            return pd.read_parquet(local_parquet_path(table_name))
        raise

NOTEBOOK_DIR = Path("data_analysis") if Path("data_analysis").is_dir() else Path(".")


In [2]:
repository_df = read_parquet_table("repository")
source_markdown_file_history_df = read_parquet_table("source_markdown_file_history")
source_markdown_file_snapshot_df = read_parquet_table("source_markdown_file_snapshot")
source_markdown_file_version_df = read_parquet_table("source_markdown_file_version")
lock_file_snapshot_df = read_parquet_table("lock_file_snapshot")


In [3]:
repository_df


,repository_id,url,license,repo_full_name,repo_owner,repo_name,main_language,forks,stars,created_at,updated_at,pushed_at
0,9089213639968762207,https://github.com/apache/cloudstack,Apache-2.0,apache/cloudstack,apache,cloudstack,Java,1334,2955,2013-04-29T22:27:12Z,2026-06-27T18:09:29Z,2026-06-26T18:15:59Z
1,4902717555605226939,https://github.com/brunoborges/fx2048,GPL-3.0,brunoborges/fx2048,brunoborges,fx2048,Java,127,291,2014-03-31T04:08:25Z,2026-06-14T06:40:51Z,2026-06-14T06:40:46Z
2,8401317167309685863,https://github.com/vaadin/flow,Apache-2.0,vaadin/flow,vaadin,flow,Java,208,745,2015-04-29T17:54:35Z,2026-06-27T17:27:01Z,2026-06-26T15:41:28Z
3,1689116779471759733,https://github.com/Activiti/activiti-cloud,Apache-2.0,Activiti/activiti-cloud,Activiti,activiti-cloud,Java,48,90,2020-01-27T11:13:42Z,2026-06-26T18:48:10Z,2026-06-27T12:44:15Z
4,8707166185860304590,https://github.com/duckduckgo/Android,Apache-2.0,duckduckgo/Android,duckduckgo,Android,Kotlin,1358,4713,2017-01-13T17:11:25Z,2026-06-26T23:05:16Z,2026-06-27T19:40:28Z
...,...,...,...,...,...,...,...,...,...,...,...,...
257,5427182896104728152,https://github.com/oboapp/oboapp,Unlicense,oboapp/oboapp,oboapp,oboapp,TypeScript,9,13,2025-12-12T12:35:25Z,2026-06-26T16:25:57Z,2026-06-26T16:25:52Z
258,8936047867513918691,https://github.com/felipementel/GitHubCopilotD...,NaN,felipementel/GitHubCopilotDevDays-Curitiba-2026,felipementel,GitHubCopilotDevDays-Curitiba-2026,C#,1,14,2026-05-09T13:06:58Z,2026-06-11T01:10:38Z,2026-06-27T11:22:27Z
259,3451869941153973759,https://github.com/fqfqgo/clash-verge-rev,GPL-3.0,fqfqgo/clash-verge-rev,fqfqgo,clash-verge-rev,TypeScript,1,18,2026-02-22T14:23:01Z,2026-06-22T04:22:51Z,2026-06-22T05:17:06Z
260,3591656885695211674,https://github.com/microsoft/AI-Engineering-Coach,MIT,microsoft/AI-Engineering-Coach,microsoft,AI-Engineering-Coach,TypeScript,448,3098,2026-05-06T20:27:16Z,2026-06-27T20:28:01Z,2026-06-25T08:50:13Z


In [4]:
source_markdown_file_history_df


,source_markdown_file_history_id,version_count
0,1,1
1,2,4
2,3,3
3,4,2
4,5,3
...,...,...
599,600,1
600,601,1
601,602,1
602,603,1


In [5]:
source_markdown_file_snapshot_df


,source_markdown_file_snapshot_id,repository_id,path,content,frontmatter,body
0,1,9089213639968762207,.github/workflows/daily-issue-triage.md,---\ndescription: |\n Scheduled daily triage ...,description: |\n Scheduled daily triage that ...,\n# Daily Issue Triage\n\n<!-- Note - this fil...
1,2,9089213639968762207,.github/workflows/daily-repo-status.md,---\ndescription: |\n This workflow creates d...,description: |\n This workflow creates daily ...,\n# Daily Repo Status\n\nCreate an upbeat dail...
2,3,9089213639968762207,.github/workflows/daily-repo-status.md,---\ndescription: |\n This workflow creates d...,description: |\n This workflow creates daily ...,\n# Daily Repo Status\n\nCreate an upbeat dail...
3,4,9089213639968762207,.github/workflows/daily-repo-status.md,---\ndescription: |\n This workflow creates d...,description: |\n This workflow creates daily ...,\n# Repo Status\n\nCreate an upbeat daily stat...
4,5,9089213639968762207,.github/workflows/daily-repo-status.md,---\ndescription: |\n This workflow creates d...,description: |\n This workflow creates daily ...,\n# Repo Status\n\nCreate an upbeat daily stat...
...,...,...,...,...,...,...
2815,2816,6077468482482437617,.github/workflows/pr-prior-art-reviewer.md,---\non:\n workflow_call:\n inputs:\n ...,on:\n workflow_call:\n inputs:\n pr_u...,\n# pr-prior-art-reviewer\n\nYou are pr-prior-...
2816,2817,6077468482482437617,.github/workflows/pr-security-reviewer.md,---\non:\n workflow_call:\n inputs:\n ...,on:\n workflow_call:\n inputs:\n pr_u...,\n# pr-security-reviewer\n\nYou are pr-securit...
2817,2818,6077468482482437617,.github/workflows/pr-testing-reviewer.md,---\non:\n workflow_call:\n inputs:\n ...,on:\n workflow_call:\n inputs:\n pr_u...,\n# pr-testing-reviewer\n\nYou are pr-testing-...
2818,2819,6077468482482437617,.github/workflows/validate-yaml-snippets.md,---\ndescription: Validates YAML snippets in m...,description: Validates YAML snippets in markdo...,\n# YAML Snippet Validator\n\nYou are an AI ag...


In [6]:
source_markdown_file_version_df


,source_markdown_file_version_id,source_markdown_file_history_id,source_markdown_file_snapshot_id,commit_sha,committed_at,rank,predecessor_source_markdown_file_version_id,successor_source_markdown_file_version_id
0,1,1,1,957bfbb1cddd1ffac718a2d3f135ded4a85b73d1,2026-06-17T18:15:03Z,1,NaN,NaN
1,2,2,2,a1bcae921367eb8c38d0e119124416f3c9fa75ff,2026-02-17T15:11:01Z,1,NaN,3.0
2,3,2,3,c0db75b9fa6bdaff31297ee0ebe29d7ce1ef8459,2026-02-18T08:35:17Z,2,2.0,4.0
3,4,2,4,7308dad19a9a70d33a33a75bea8c51629541e438,2026-05-29T09:15:49Z,3,3.0,5.0
4,5,2,5,957bfbb1cddd1ffac718a2d3f135ded4a85b73d1,2026-06-17T18:15:03Z,4,4.0,NaN
...,...,...,...,...,...,...,...,...
2815,2816,601,2816,0611e9dbcf7793711e5a3adf8a0fd2329c83f779,2026-06-05T16:40:00Z,1,NaN,NaN
2816,2817,602,2817,0611e9dbcf7793711e5a3adf8a0fd2329c83f779,2026-06-05T16:40:00Z,1,NaN,NaN
2817,2818,603,2818,0611e9dbcf7793711e5a3adf8a0fd2329c83f779,2026-06-05T16:40:00Z,1,NaN,NaN
2818,2819,604,2819,82f70fcd39635b14f151bf770431c85c6d1eaad6,2026-01-15T23:14:12Z,1,NaN,2820.0


In [7]:
lock_file_snapshot_df


,lock_file_snapshot_id,source_markdown_file_version_id,path,content,file_sha
0,1,1,.github/workflows/daily-issue-triage.lock.yml,"# gh-aw-metadata: {""schema_version"":""v3"",""fron...",bd7db9978d40bfd8799dc53087f916a9ac5691a8
1,2,2,.github/workflows/daily-repo-status.lock.yml,#\n# ___ _ _\n# / _ \...,36847a060a10d0e585b1abffbd6e4e3eea3a3ba9
2,3,3,.github/workflows/daily-repo-status.lock.yml,#\n# ___ _\n# / _ \ ...,1d7e7eecd14dee89841a371b7d85ea5dd9032e19
3,4,4,.github/workflows/daily-repo-status.lock.yml,"# gh-aw-metadata: {""schema_version"":""v3"",""fron...",7c3d20a166ea1080045dfd7a99e3d638900f5fa0
4,5,5,.github/workflows/daily-repo-status.lock.yml,"# gh-aw-metadata: {""schema_version"":""v3"",""fron...",98d26f1d559e5ccb0f3719d863a084d24777d877
...,...,...,...,...,...
2815,2816,2816,.github/workflows/pr-prior-art-reviewer.lock.yml,"# gh-aw-metadata: {""schema_version"":""v3"",""fron...",0430f792a7c0f2320b8010545494e21f3f9606f5
2816,2817,2817,.github/workflows/pr-security-reviewer.lock.yml,"# gh-aw-metadata: {""schema_version"":""v3"",""fron...",b8b99b6daf80020c24f983e47afb95f8cc3b0399
2817,2818,2818,.github/workflows/pr-testing-reviewer.lock.yml,"# gh-aw-metadata: {""schema_version"":""v3"",""fron...",c15f52d374d27bba37ff9a49f9656de46ae50894
2818,2819,2819,.github/workflows/validate-yaml-snippets.lock.yml,#\n# ___ _ _ \n# ...,fc6242d38079d9928724d3de9257a6f9aed58df7


## Table sizes

In [8]:
table_sizes = pd.DataFrame(
    [
        {"table": "repository", "rows": len(repository_df)},
        {
            "table": "source_markdown_file_history",
            "rows": len(source_markdown_file_history_df),
        },
        {
            "table": "source_markdown_file_snapshot",
            "rows": len(source_markdown_file_snapshot_df),
        },
        {
            "table": "source_markdown_file_version",
            "rows": len(source_markdown_file_version_df),
        },
        {"table": "lock_file_snapshot", "rows": len(lock_file_snapshot_df)},
    ]
)
table_sizes


,table,rows
0,repository,262
1,source_markdown_file_history,604
2,source_markdown_file_snapshot,2820
3,source_markdown_file_version,2820
4,lock_file_snapshot,2820


## Join source versions with repositories

This dataset does not have pull requests or users like AIDev. The central history view links repositories, observed source Markdown snapshots, and source Markdown versions.

In [9]:
source_version_repo_df = (
    source_markdown_file_version_df.merge(
        source_markdown_file_snapshot_df,
        on="source_markdown_file_snapshot_id",
        how="inner",
        validate="many_to_one",
    )
    .merge(
        repository_df,
        on="repository_id",
        how="left",
        validate="many_to_one",
    )
)

source_version_repo_df


,source_markdown_file_version_id,source_markdown_file_history_id,source_markdown_file_snapshot_id,commit_sha,committed_at,rank,predecessor_source_markdown_file_version_id,successor_source_markdown_file_version_id,repository_id,path,content,frontmatter,body,url,license,repo_full_name,repo_owner,repo_name,main_language,forks,stars,created_at,updated_at,pushed_at
0,1,1,1,957bfbb1cddd1ffac718a2d3f135ded4a85b73d1,2026-06-17T18:15:03Z,1,NaN,NaN,9089213639968762207,.github/workflows/daily-issue-triage.md,---\ndescription: |\n Scheduled daily triage ...,description: |\n Scheduled daily triage that ...,\n# Daily Issue Triage\n\n<!-- Note - this fil...,https://github.com/apache/cloudstack,Apache-2.0,apache/cloudstack,apache,cloudstack,Java,1334,2955,2013-04-29T22:27:12Z,2026-06-27T18:09:29Z,2026-06-26T18:15:59Z
1,2,2,2,a1bcae921367eb8c38d0e119124416f3c9fa75ff,2026-02-17T15:11:01Z,1,NaN,3.0,9089213639968762207,.github/workflows/daily-repo-status.md,---\ndescription: |\n This workflow creates d...,description: |\n This workflow creates daily ...,\n# Daily Repo Status\n\nCreate an upbeat dail...,https://github.com/apache/cloudstack,Apache-2.0,apache/cloudstack,apache,cloudstack,Java,1334,2955,2013-04-29T22:27:12Z,2026-06-27T18:09:29Z,2026-06-26T18:15:59Z
2,3,2,3,c0db75b9fa6bdaff31297ee0ebe29d7ce1ef8459,2026-02-18T08:35:17Z,2,2.0,4.0,9089213639968762207,.github/workflows/daily-repo-status.md,---\ndescription: |\n This workflow creates d...,description: |\n This workflow creates daily ...,\n# Daily Repo Status\n\nCreate an upbeat dail...,https://github.com/apache/cloudstack,Apache-2.0,apache/cloudstack,apache,cloudstack,Java,1334,2955,2013-04-29T22:27:12Z,2026-06-27T18:09:29Z,2026-06-26T18:15:59Z
3,4,2,4,7308dad19a9a70d33a33a75bea8c51629541e438,2026-05-29T09:15:49Z,3,3.0,5.0,9089213639968762207,.github/workflows/daily-repo-status.md,---\ndescription: |\n This workflow creates d...,description: |\n This workflow creates daily ...,\n# Repo Status\n\nCreate an upbeat daily stat...,https://github.com/apache/cloudstack,Apache-2.0,apache/cloudstack,apache,cloudstack,Java,1334,2955,2013-04-29T22:27:12Z,2026-06-27T18:09:29Z,2026-06-26T18:15:59Z
4,5,2,5,957bfbb1cddd1ffac718a2d3f135ded4a85b73d1,2026-06-17T18:15:03Z,4,4.0,NaN,9089213639968762207,.github/workflows/daily-repo-status.md,---\ndescription: |\n This workflow creates d...,description: |\n This workflow creates daily ...,\n# Repo Status\n\nCreate an upbeat daily stat...,https://github.com/apache/cloudstack,Apache-2.0,apache/cloudstack,apache,cloudstack,Java,1334,2955,2013-04-29T22:27:12Z,2026-06-27T18:09:29Z,2026-06-26T18:15:59Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2815,2816,601,2816,0611e9dbcf7793711e5a3adf8a0fd2329c83f779,2026-06-05T16:40:00Z,1,NaN,NaN,6077468482482437617,.github/workflows/pr-prior-art-reviewer.md,---\non:\n workflow_call:\n inputs:\n ...,on:\n workflow_call:\n inputs:\n pr_u...,\n# pr-prior-art-reviewer\n\nYou are pr-prior-...,https://github.com/drasi-project/drasi-server,Apache-2.0,drasi-project/drasi-server,drasi-project,drasi-server,Rust,7,10,2025-08-25T21:33:50Z,2026-06-19T02:02:41Z,2026-06-26T03:10:53Z
2816,2817,602,2817,0611e9dbcf7793711e5a3adf8a0fd2329c83f779,2026-06-05T16:40:00Z,1,NaN,NaN,6077468482482437617,.github/workflows/pr-security-reviewer.md,---\non:\n workflow_call:\n inputs:\n ...,on:\n workflow_call:\n inputs:\n pr_u...,\n# pr-security-reviewer\n\nYou are pr-securit...,https://github.com/drasi-project/drasi-server,Apache-2.0,drasi-project/drasi-server,drasi-project,drasi-server,Rust,7,10,2025-08-25T21:33:50Z,2026-06-19T02:02:41Z,2026-06-26T03:10:53Z
2817,2818,603,2818,0611e9dbcf7793711e5a3adf8a0fd2329c83f779,2026-06-05T16:40:00Z,1,NaN,NaN,6077468482482437617,.github/workflows/pr-testing-reviewer.md,---\non:\n workflow_call:\n inputs:\n ...,on:\n workflow_call:\n inputs:\n pr_u...,\n# pr-testing-reviewer\n\nYou are pr-testing-...,https://github.com/drasi-project/drasi-server,Apache-2.0,drasi-pr

## Observed adoption by repository

A repository's observed adoption date is the earliest `committed_at` timestamp among its published GH-AW source Markdown versions.

In [10]:
source_version_repo_df["committed_at"] = pd.to_datetime(
    source_version_repo_df["committed_at"],
    utc=True,
    errors="coerce",
)

observed_adoption_df = (
    source_version_repo_df.dropna(subset=["committed_at"])
    .sort_values(["repository_id", "committed_at", "rank"])
    .groupby("repository_id", as_index=False)
    .agg(
        repo_full_name=("repo_full_name", "first"),
        main_language=("main_language", "first"),
        license=("license", "first"),
        stars=("stars", "first"),
        forks=("forks", "first"),
        first_observed_at=("committed_at", "first"),
        first_source_path=("path", "first"),
        observed_versions=("source_markdown_file_version_id", "count"),
        observed_source_paths=("path", "nunique"),
    )
    .sort_values("first_observed_at")
)

observed_adoption_df


,repository_id,repo_full_name,main_language,license,stars,forks,first_observed_at,first_source_path,observed_versions,observed_source_paths
211,7640649877921565393,microsoft/wassette,Rust,MIT,917,69,2025-10-03 20:05:08+00:00,.github/workflows/issue-triage.md,11,8
23,892592735414390340,githubnext/agentics,Makefile,MIT,811,115,2025-10-22 18:14:56+00:00,.github/workflows/maintainer.md,44,5
198,7061224288023129471,kaito-project/aikit,Go,MIT,529,57,2025-10-24 18:03:50+00:00,.github/workflows/issue-triage.md,3,2
97,3532360678775308556,appwrite/appwrite,TypeScript,BSD-3-Clause,56420,5488,2025-10-28 03:00:30+00:00,.github/workflows/issue-triage.md,13,1
163,6001044222039231227,lablup/backend.ai-webui,TypeScript,LGPL-3.0,130,80,2025-11-25 09:26:02+00:00,.github/workflows/daily-test-improver.md,15,5
...,...,...,...,...,...,...,...,...,...,...
103,3721005143461110050,popey/slomore,Rust,MIT,27,2,2026-06-02 16:08:18+00:00,.github/workflows/repo-assist.md,3,1
154,5423802872524571249,popey/sbom-vm,Python,MIT,19,4,2026-06-02 16:18:52+00:00,.github/workflows/repo-assist.md,4,1
232,8401317167309685863,vaadin/flow,Java,Apache-2.0,745,208,2026-06-04 08:47:29+00:00,.github/workflows/doc-bot.md,2,1
28,1084606090191205130,dgreif/ring,TypeScript,MIT,1499,201,2026-06-06 11:44:57+00:00,.github/workflows/issue-triage.md,2,1


## Monthly cumulative adoption

In [11]:
monthly_adoption_df = (
    observed_adoption_df.assign(
        adoption_month=lambda df: df["first_observed_at"]
        .dt.tz_convert(None)
        .dt.to_period("M")
        .dt.to_timestamp()
    )
    .groupby("adoption_month", as_index=False)
    .agg(new_repositories=("repository_id", "count"))
    .sort_values("adoption_month")
)
monthly_adoption_df["cumulative_repositories"] = monthly_adoption_df[
    "new_repositories"
].cumsum()

monthly_adoption_df


,adoption_month,new_repositories,cumulative_repositories
0,2025-10-01,4,4
1,2025-11-01,1,5
2,2025-12-01,3,8
3,2026-01-01,6,14
4,2026-02-01,67,81
5,2026-03-01,62,143
6,2026-04-01,35,178
7,2026-05-01,75,253
8,2026-06-01,9,262
